# Reproducing Chandravadia et al. (2020) from dandiset 000004

Paper: Chandravadia, Liang, Schjetnan et al. (2020), "A NWB-based dataset and processing pipeline of
human single-neuron activity during a declarative memory task", Scientific Data 7, 78,
doi:10.1038/s41597-020-0415-9.

Dataset: DANDI:000004, version **0.220126.1852** (doi:10.48324/dandi.000004/0.220126.1852). This version
was asked for by the run's prompt; it is the version the claims file was written against, and the
dandiset's most recent published version. The version published two seconds before it, 0.220126.1851,
holds the same 87 files (same paths, blobs and sizes, shown in the data check), and so does the draft. The
files are streamed from the archive; the stimulus images are not read.

The claims file (`claims/chandravadia-2020.yaml`) was written from the open access full text of the
version of record (Methods, Data Records, Technical Validation, legends, Tables 1 and 2 and Online-only
Table 1). Method details it lacks are taken from the paper's code,
https://github.com/rutishauserlab/recogmem-release-NWB at commit 1d95162d2672e71dd234a31e4b8d70017a289ee3,
as cited in the helper module `recogmem.py`.

Outside inputs, read at run time from Europe PMC's copy of the open access full texts: Online-only
Table 1 of this paper (PMC7055261), the reference list of sessions and subjects for several claims;
and Table 1 of Faraut et al. 2018 (doi:10.1038/sdata.2018.10, PMC5810422), the earlier release, for
the two `with_inputs` claims. For fig3-example-vs-neuron the file list of the paper's own
deposit on OSF (doi:10.17605/OSF.IO/HV7JA) is searched for session 144, and the paper's Figure 3 scripts
are read at the pinned commit for the session they name.

| Claim | Tested here |
|---|---|
| counts-neurons-subjects | tested |
| counts-sessions | tested |
| table1-demographics | tested |
| task-structure | tested |
| new-old-label-coding | tested |
| ttl-codes | tested |
| experiment-ids | tested |
| session-dates | tested |
| units-table | tested |
| electrodes-table | tested |
| isi-criterion | tested |
| fig5a-mtl-locations | tested |
| fig5c-auc-all | tested |
| fig5c-auc-sites | tested |
| fig5b-roc-asymmetric | tested |
| fig5d-confidence-accuracy | tested |
| vs-cells-proportion | tested |
| ms-cells-proportion | tested |
| dual-cells-proportion | tested |
| fig5f-site-proportions | tested |
| twh-sessions | tested |
| previous-release-link | tested (with Faraut et al. 2018, Table 1) |
| previous-release-additions | tested (with Faraut et al. 2018, Table 1) |
| fig4-example-vs-neuron | tested |
| fig3-example-vs-neuron | tested, on session 2009 (TWH103), which the paper's code names as session 144 |

In [ ]:
import collections, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import recogmem as rm

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)
t0 = time.time()
assets = rm.list_assets()
S = rm.load_all([(p, u) for p, u, _ in assets])
SID = {s["session_id"]: s for s in S}
print(f"{len(assets)} NWB files, {sum(a[2] for a in assets)/1e9:.2f} GB on the archive; loaded in {time.time()-t0:.0f} s")

## Data check

In [ ]:
meta = rm.dandiset_metadata()
print("Dandiset:", meta["name"])
print("Version:", meta["version"], " DOI:", meta.get("doi"))
print("Archive summary:", {k: meta["assetsSummary"].get(k) for k in ["numberOfFiles", "numberOfSubjects", "numberOfBytes"]})
print("Related:", [(r.get("relation"), r.get("identifier")) for r in meta.get("relatedResource", [])])
va = {v: rm.version_assets(v) for v in ["0.220126.1851", rm.DANDISET_VERSION]}
print(f"Published versions 0.220126.1851 and {rm.DANDISET_VERSION}: {len(va['0.220126.1851'])} and {len(va[rm.DANDISET_VERSION])} assets;"
      f" same paths, blob ids and sizes: {va['0.220126.1851'] == va[rm.DANDISET_VERSION]}")

paper_t1 = rm.patient_table(rm.PAPER_PMC, "Online-only Table 1")
faraut_t1 = rm.patient_table(rm.FARAUT_PMC, "Table 1")
print(f"Online-only Table 1 (this paper): {paper_t1.label.nunique()} subjects, {len(paper_t1)} sessions")
print(f"Table 1 of Faraut et al. 2018: {faraut_t1.label.nunique()} subjects, {len(faraut_t1)} sessions")

rows = []
for s in S:
    t = s["trials"]
    r = rm.recog_trials(s)
    rows.append({
        "identifier": s["identifier"], "label": s["label"],
        "table_label": rm.table_label(s["subject"]["subject_id"]), "session": s["session_id"],
        "site": rm.site(s["label"]), "start": s["session_start_time"][:10],
        "learn": (t.stim_phase == "learn").sum(), "recog": (t.stim_phase == "recog").sum(),
        "old(1)": (r.label == 1).sum(), "new(0)": (r.label == 0).sum(), "cats": t.stimCategory.nunique(),
        "units": len(s["unit_ids"]), "electrodes": len(s["electrodes"]),
        "exp_ids": tuple(int(x) for x in np.unique(s["expids"])), "variant": rm.variant_from_images(s),
    })
F = pd.DataFrame(rows)
print(F.iloc[:44].to_string())

In [ ]:
print(F.iloc[44:].to_string())
print("File names:", [s["path"] for s in S])

In [ ]:
# The patient tables as parsed from Europe PMC, one row per (subject, session), to check against the PDF tables.
print("Online-only Table 1 of this paper, parsed:")
print(paper_t1.to_string())
print("Table 1 of Faraut et al. 2018, parsed:")
print(faraut_t1.to_string())

In [ ]:
print("Files per site:", F.site.value_counts().to_dict(), " subjects per site:", F.groupby("site").label.nunique().to_dict())
print("Units per site:", F.groupby("site").units.sum().to_dict(), " total", F.units.sum())
print("Identifier label agrees with subject_id:", (F.label.str.replace("CS", "C") == F.table_label).all())
print("Session id in identifier equals the one in session_description:",
      all(s["session_id"] == s["session_id_description"] for s in S))
print("Trials table columns:", collections.Counter(tuple(s["trials_columns"]) for s in S).most_common())
print("Units table columns:", collections.Counter(tuple(s["units_columns"]) for s in S).most_common())
print("Electrodes table columns:", collections.Counter(tuple(s["electrodes_columns"]) for s in S).most_common())
print("Stimulus series (name, shape):", collections.Counter(str(s["stimulus_series"]) for s in S).most_common())
print("NWB versions:", collections.Counter(s["nwb_version"] for s in S))

R = pd.concat([rm.recog_trials(s).assign(session=s["session_id"]) for s in S])
print("Recognition response_value - 30:", dict(sorted(collections.Counter(R.conf).items())))
L = pd.concat([s["trials"][s["trials"].stim_phase == "learn"] for s in S])
print("Learning response_value:", dict(collections.Counter(L.response_value)))
print("new_old_labels_recog in learning rows:", dict(collections.Counter(L.new_old_labels_recog)))
mism = [s["session_id"] for s in S if not np.array_equal(rm.check_responses_against_events(s), rm.recog_trials(s).conf.values)]
print("Sessions where response_value - 30 differs from the event codes 30 to 36 the paper's code reads:", mism)

allt = pd.concat([s["trials"] for s in S])
print("start_time == stim_on_time in every row:", np.allclose(allt.start_time, allt.stim_on_time))
print("stop_time == delay2_time (end of trial) in %.3f of rows; stop_time == stim_off_time in %.3f" %
      (np.mean(np.isclose(allt.stop_time, allt.delay2_time)), np.mean(np.isclose(allt.stop_time, allt.stim_off_time))))
dur = allt.stim_off_time - allt.stim_on_time
print("Stimulus duration stim_off_time - stim_on_time (s): %.3f to %.3f; within 0.1 s of 1 s: %d, of 2 s: %d trials" %
      (dur.min(), dur.max(), np.sum(np.abs(dur - 1) < 0.1), np.sum(np.abs(dur - 2) < 0.1)))
epoch = [s["session_id"] for s in S if s["trials"].start_time.min() > 1e8]
print(f"Sessions whose times are Unix timestamps (seconds since 1970) in place of seconds from the start of the recording: {len(epoch)}", epoch)
rt = R.response_time - R.delay1_time
print("Recognition response after question onset (s): median %.2f, range %.2f to %.2f" % (rt.median(), rt.min(), rt.max()))

In [ ]:
# Clocks: trial times against the event timestamps, and spikes against trials.
rows = []
for s in S:
    t = s["trials"]
    on_ev = s["events_t"][s["events"] == 1]
    match = np.isin(np.round(t.start_time.values, 6), np.round(on_ev, 6)).mean()
    spk = np.concatenate(s["spikes"])
    inside = np.mean((t.start_time.values >= spk.min()) & (t.start_time.values <= spk.max()))
    rows.append({"session": s["session_id"], "onsets_in_events": match, "onsets_within_spikes": inside})
C = pd.DataFrame(rows)
print("Fraction of trial start_time values found among the timestamps of event code 1: min %.3f, median %.3f" %
      (C.onsets_in_events.min(), C.onsets_in_events.median()))
print("Sessions where not all trial onsets are event timestamps:", C[C.onsets_in_events < 1].session.tolist())
print("Fraction of trial onsets between the first and last spike of the session: min %.3f; sessions below 1:" % C.onsets_within_spikes.min(),
      C[C.onsets_within_spikes < 1][["session", "onsets_within_spikes"]].round(2).values.tolist())

The spikes share the clock of the trials table if units respond to stimulus onset and to the stimulus
category at the stated onsets and not at shifted ones. The test: for each unit, the rate 0.2 to 1.2 s
after onset against 1 s before onset (Wilcoxon signed-rank over the 100 recognition trials), and the
VS ANOVA, with the true onsets and with all onsets shifted by +3 s and -3 s (into the inter-trial
period). The shift is a standard choice for this check.

In [ ]:
rows = []
for s in S:
    r = rm.recog_trials(s)
    on, cat = r.start_time.values, r.stimCategory.values
    for i, sp in enumerate(s["spikes"]):
        d = {"session": s["session_id"]}
        for name, shift in [("true", 0.0), ("+3s", 3.0), ("-3s", -3.0)]:
            post = rm.window_rates(sp, on + shift, rm.METHODS_WINDOW)
            pre = rm.window_rates(sp, on + shift, (-1.0, 0.0))
            diff = post - pre
            d["resp_" + name] = stats.wilcoxon(post, pre).pvalue < 0.05 if np.any(diff != 0) else False
            d["vs_" + name] = rm.vs_p(post, cat) < 0.05
        rows.append(d)
CK = pd.DataFrame(rows)
print("Units responsive to onset (p<0.05):", {k: int(CK["resp_" + k].sum()) for k in ["true", "+3s", "-3s"]}, "of", len(CK))
print("Units category selective (p<0.05):", {k: int(CK["vs_" + k].sum()) for k in ["true", "+3s", "-3s"]})
per = CK.groupby("session")[["resp_true", "resp_+3s", "resp_-3s"]].sum()
better = (per["resp_true"] > per[["resp_+3s", "resp_-3s"]].max(axis=1))
print(f"Sessions with more responsive units at the true onsets than at either shift: {better.sum()} of {len(per)}")
print("Sessions where this does not hold (units, responsive true/+3/-3):")
print(per[~better].join(F.set_index("session").units).to_string())

In [ ]:
drop, pairs = rm.duplicated_trains(S)
print("Spike trains identical to a train of an earlier file:", len(drop))
for (a, b), n in pairs.items():
    sa, sb = SID[a], SID[b]
    same_events = len(sa["events"]) == len(sb["events"]) and np.array_equal(sa["events"], sb["events"]) and np.allclose(sa["events_t"], sb["events_t"])
    ta, tb = sa["trials"], sb["trials"]
    same_trials = len(ta) == len(tb) and np.allclose(ta.start_time, tb.start_time)
    same_imgs = len(ta) == len(tb) and (ta.external_image_file.values == tb.external_image_file.values).all()
    ra = (ta.start_time.min(), ta.stop_time.max()); rb = (tb.start_time.min(), tb.stop_time.max())
    print(f"  sessions {a} ({sa['label']}, {len(sa['spikes'])} units) and {b} ({sb['label']}, {len(sb['spikes'])} units): "
          f"trials of {a} {ra[0]:.0f} to {ra[1]:.0f} s, of {b} {rb[0]:.0f} to {rb[1]:.0f} s; "
          f"{n} identical trains; identical events {same_events}, identical trial onsets {same_trials}, identical image list {same_imgs}")
    if len(ta) == len(tb):
        dt = (tb.start_time.values - ta.start_time.values)
        print(f"     onset difference b - a: {dt.min():.3f} to {dt.max():.3f} s; responses equal in {np.mean(ta.response_value.values == tb.response_value.values):.2f} of trials")

## counts-neurons-subjects

> The dataset holds 1863 single neurons recorded from the medial temporal lobes of 59 subjects.

Criterion: neurons are the unit rows of all files, subjects the distinct subject_id values; reading
(a) every unit row, (b) spike trains deposited in two files counted once.

In [ ]:
n_units = sum(len(s["unit_ids"]) for s in S)
n_subj = len({s["subject"]["subject_id"] for s in S})
print(f"(a) unit rows: {n_units} (expected 1863 +/- 186.3, difference {n_units-1863:+d}); subjects: {n_subj} (expected 59 +/- 5.9)")
print(f"(b) trains counted once: {n_units-len(drop)} (difference {n_units-len(drop)-1863:+d})")
print("Units per subject: min %d, median %d, max %d" % tuple(F.groupby("label").units.sum().agg(["min", "median", "max"])))
print("Dandiset metadata numberOfSubjects:", meta["assetsSummary"]["numberOfSubjects"])

**Result: reproduced.** The files hold 1864 unit rows (one more than the paper's 1863) from 59 subjects,
both within tolerance under reading (a). Counting the 66 spike trains that appear in two files once
gives 1798, also within tolerance.

## counts-sessions

> The patients performed 87 sessions, listed by subject and session id in Online-only Table 1 (the Methods give 89).

Criterion: the set of (subject, session id) pairs in the files against Online-only Table 1.

In [ ]:
files_set = set(zip(F.table_label, F.session))
table_set = set(zip(paper_t1.label, paper_t1.session_id))
print(f"Files: {len(files_set)} (subject, session) pairs; Online-only Table 1: {len(table_set)}; Methods text: 89")
print("In the files, not in the table:", sorted(files_set - table_set))
print("In the table, not in the files:", sorted(table_set - files_set))
v = paper_t1.set_index("session_id").variant
F["table_variant"] = F.session.map(v)
print("Variant from the image directories equals the table's variant in", (F.variant == F.table_variant).sum(), "of", len(F), "sessions")
print(F[F.variant != F.table_variant][["session", "table_label", "variant", "table_variant"]].to_string())

**Result: reproduced.** The 87 (subject, session id) pairs of the files are exactly those of Online-only
Table 1, and the task variant read from the image directories equals the table's in all 87. The
Methods' 89 sessions is not what the files hold.

## table1-demographics

> Online-only Table 1 gives the age and sex of every subject; there are 25 female and 34 male subjects.

In [ ]:
subj = pd.DataFrame([{"table_label": rm.table_label(s["subject"]["subject_id"]), "file_age": s["subject"].get("age"),
                      "file_sex": s["subject"].get("sex"), "file_diagnosis": s["subject"].get("description"),
                      "session": s["session_id"]} for s in S])
per_subj = subj.groupby("table_label").agg(file_age=("file_age", lambda x: sorted(set(x))), file_sex=("file_sex", lambda x: sorted(set(x))),
                                           file_diagnosis=("file_diagnosis", lambda x: sorted(set(x))))
tab = paper_t1.groupby("label").first()[["age", "sex", "diagnosis"]]
cmp_ = per_subj.join(tab, how="outer")
cmp_["age_ok"] = [len(a) == 1 and str(a[0]) == str(b) for a, b in zip(cmp_.file_age, cmp_.age)]
cmp_["sex_ok"] = [len(a) == 1 and a[0] == b for a, b in zip(cmp_.file_sex, cmp_.sex)]
print(f"Subjects: {len(cmp_)}; age matches {cmp_.age_ok.sum()}, sex matches {cmp_.sex_ok.sum()}, both {(cmp_.age_ok & cmp_.sex_ok).sum()}")
print("Subjects that differ:")
print(cmp_[~(cmp_.age_ok & cmp_.sex_ok)][["file_age", "age", "file_sex", "sex"]].to_string())
fs = per_subj.file_sex.map(lambda x: x[0]).value_counts().to_dict()
print("Files, subjects by sex:", fs, "  table:", tab.sex.value_counts().to_dict(), " (expected F 25, M 34)")
print("Diagnosis, file against table (first 12 subjects):")
print(cmp_[["file_diagnosis", "diagnosis"]].head(12).to_string())
print("Distinct diagnosis values in files:", sorted({d for x in per_subj.file_diagnosis for d in x}))
print("Diagnosis text equal to the table's in", sum(len(a) == 1 and a[0] == b for a, b in zip(cmp_.file_diagnosis, cmp_.diagnosis)), "of", len(cmp_))

**Result: partially reproduced.** Age and sex match Online-only Table 1 for 56 of 59 subjects. C24 is 57
in the files and 47 in the table, H33 30 and 29, and H48 is 32 and female in the files, 54 and male in
the table; Faraut et al. 2018 gives the same values as the table for these three (previous-release-link).
The files count 26 female and 33 male subjects, against 25 and 34. The diagnosis in the files is a
coarser coding; its text equals the table's for 40 of 59 subjects.

## task-structure

> Each session has 100 encoding trials showing 100 novel images and 100 recognition trials, of which 50
> show new and 50 old images, and the images come from 5 visual categories.

In [ ]:
rows = []
for s in S:
    t = s["trials"]
    le, rc = t[t.stim_phase == "learn"], t[t.stim_phase == "recog"]
    rows.append({"session": s["session_id"], "learn": len(le), "learn_distinct_images": le.external_image_file.nunique(),
                 "recog": len(rc), "label0": (rc.new_old_labels_recog == "0").sum(), "label1": (rc.new_old_labels_recog == "1").sum(),
                 "categories": t.stimCategory.nunique(), "images_stored": list(s["stimulus_series"].values())[0][0]})
T = pd.DataFrame(rows)
T["enc_ok"] = (T.learn == 100) & (T.learn_distinct_images == 100)
T["rec_ok"] = (T.recog == 100) & (T.label0 == 50) & (T.label1 == 50)
T["cat_ok"] = T.categories == 5
print("Sessions with 100 encoding trials of 100 distinct images:", T.enc_ok.sum(), "of", len(T))
print("Sessions with 100 recognition trials, 50 of each label:", T.rec_ok.sum())
print("Sessions with 5 categories:", T.cat_ok.sum())
print("Sessions with stored images equal to the number of trials:", (T.images_stored == T.learn + T.recog).sum())
print(T[~(T.enc_ok & T.rec_ok & T.cat_ok)].to_string())

**Result: partially reproduced.** All 87 sessions have 100 recognition trials with 50 of each label and
5 categories, but 8 sessions have fewer encoding trials: 7 have 50 (18, 20, 21, 23, 26, 27, 44) and
session 123 has 97. Each part holds in more than half of the sessions.

## new-old-label-coding

> In the trials table, new_old_labels_recog is 0 for old stimuli (seen during encoding) and 1 for new stimuli.

Criterion: per session, (1) whether label-0 trials show images of the session's encoding block and
label-1 trials do not, and (2) whether label-0 trials receive "old" responses (4 to 6) more often.

In [ ]:
rows = []
for s in S:
    t = s["trials"]
    enc = set(t[t.stim_phase == "learn"].external_image_file)
    r = rm.recog_trials(s)
    seen = r.external_image_file.isin(enc)
    oldresp = r.conf >= 4
    rows.append({"session": s["session_id"],
                 "label0_seen": seen[r.label == 0].mean(), "label1_seen": seen[r.label == 1].mean(),
                 "label0_oldresp": oldresp[r.label == 0].mean(), "label1_oldresp": oldresp[r.label == 1].mean()})
N = pd.DataFrame(rows)
stated = (N.label0_seen == 1) & (N.label1_seen == 0) & (N.label0_oldresp > N.label1_oldresp)
reversed_ = (N.label1_seen == 1) & (N.label0_seen == 0) & (N.label1_oldresp > N.label0_oldresp)
print(f"Both tests hold under the stated coding (0 = old) in {stated.sum()} of {len(N)} sessions; with the labels reversed in {reversed_.sum()}")
print("Fraction of label-0 trials showing an encoding image: %.2f to %.2f; label-1: %.2f to %.2f" %
      (N.label0_seen.min(), N.label0_seen.max(), N.label1_seen.min(), N.label1_seen.max()))
print("Fraction of 'old' responses, label 0: median %.2f; label 1: median %.2f" % (N.label0_oldresp.median(), N.label1_oldresp.median()))
print("Sessions where label 1 gets fewer 'old' responses than label 0:", N[N.label1_oldresp <= N.label0_oldresp].session.tolist())
print("Sessions where not every label-1 image is in the encoding block (fraction):", N[N.label1_seen < 1][["session", "label1_seen"]].round(2).values.tolist())
print("Column description in the files:", collections.Counter(s["label_description"] for s in S).most_common())

**Result: not reproduced.** The labels are reversed in the files: label 0 trials never show an image of
the encoding block (fraction 0.00 in every session) and label 1 trials do, and label 1 trials receive
"old" responses more often (median 0.70 against 0.24). Both tests hold with the labels reversed in 78 of
87 sessions and under the stated coding in none. In the sessions with 50 encoding trials only part of
the label-1 images is in the encoding block (lowest 0.52), and in session 2005 label 1 does not get
more "old" responses. The column's description in the files states the same reversed coding as the
paper; the paper's code treats 1 as old.

## ttl-codes

> The events stream holds the TTL markers 55, 1, 2, 3, 20 or 21, 31 to 36, 6 and 66 with the meanings
> of Table 1, and the events and experiment_ids streams have one entry each, in the same order.

In [ ]:
listed = {55, 1, 2, 3, 20, 21, 31, 32, 33, 34, 35, 36, 6, 66}
rows = []
for s in S:
    same = len(s["events"]) == len(s["expids"]) and np.array_equal(s["events_t"], s["expids_t"])
    other = collections.Counter(c for c in s["events"] if c not in listed)
    n, bad = rm.trial_event_order(s)
    rows.append({"session": s["session_id"], "same_len_ts": same, "other_codes": dict(other), "onsets": n, "bad_order": bad,
                 "n55": int(np.sum(s["events"] == 55)), "n66": int(np.sum(s["events"] == 66))})
E = pd.DataFrame(rows)
E["ok"] = E.same_len_ts & (E.other_codes.map(len) == 0) & (E.bad_order == 0)
print("Files with equal length and timestamps of events and experiment_ids:", E.same_len_ts.sum(), "of", len(E))
print("Files holding only the listed codes:", (E.other_codes.map(len) == 0).sum())
print("Files where every trial shows 1, 2, 3, response, 6 in order:", (E.bad_order == 0).sum(), "; trials out of order:", E.bad_order.sum(), "of", E.onsets.sum())
print("Files meeting all three:", E.ok.sum(), "of", len(E))
print("Code counts over all files:", dict(sorted(collections.Counter(np.concatenate([s['events'] for s in S])).items())))
print(E[~E.ok].to_string())
print("Files with 55 not twice:", E[E.n55 != 2].session.tolist(), " with 66 not twice:", E[E.n66 != 2].session.tolist())

**Result: partially reproduced.** In all 87 files events and experiment_ids have the same length and
timestamps. 78 files meet every part of the criterion. Nine files hold codes that Table 1 does not list
(0, 4, 22, 23, 98, 99; 14 events in all), and 28 of 17047 trials in 7 files do not show the order
1, 2, 3, response, 6. Eight files do not hold the start of experiment marker (55) twice, and one holds the end marker (66) three times.

## experiment-ids

> The learning block is labelled with one of 80, 83 or 88 and the recognition block with one of 81, 84 or 89.

The learning block is the experiment id of the trials with stim_phase "learn" (the id of the events
around their onsets), the recognition block that of the "recog" trials.

In [ ]:
rows = []
for s in S:
    t = s["trials"]
    ids = []
    for ph in ["learn", "recog"]:
        on = t[t.stim_phase == ph].start_time.values
        k = np.searchsorted(s["expids_t"], on)
        k = np.clip(k, 0, len(s["expids_t"]) - 1)
        ids.append(tuple(sorted(set(s["expids"][k].astype(int)))))
    rows.append({"session": s["session_id"], "learn_id": ids[0], "recog_id": ids[1], "data_collection": s["data_collection"]})
X = pd.DataFrame(rows)
X["ok"] = X.learn_id.map(lambda x: len(x) == 1 and x[0] in (80, 83, 88)) & X.recog_id.map(lambda x: len(x) == 1 and x[0] in (81, 84, 89))
X["pair_ok"] = [(l[0], r[0]) in [(80, 81), (83, 84), (88, 89)] if len(l) == 1 and len(r) == 1 else False for l, r in zip(X.learn_id, X.recog_id)]
print("Files whose two ids fall in the stated sets:", X.ok.sum(), "of", len(X), "; whose pair is one of Table 2's:", X.pair_ok.sum())
print(collections.Counter(zip(X.learn_id, X.recog_id)))
print(X[~X.ok].to_string())
lt = [len(l) == 1 and len(r) == 1 and l[0] < r[0] for l, r in zip(X.learn_id, X.recog_id)]
print("Files whose learning trials carry the smaller of the two ids (assumed by trial_event_order and check_responses_against_events):",
      sum(lt), "of", len(X))

**Result: partially reproduced.** The experiment ids of 83 of 87 files are one of Table 2's pairs. Four
files use other ids: session 55 (80, 82), 77 (81, 82), 100 and 105 (82, 83); the data_collection field
of each file states the same ids. In all 87 files the learning trials carry the smaller of the two ids,
as the helpers that read the events assume.

## session-dates

> To protect patient information, session_start_time keeps only the year and month of the experiment,
> with the day set to the first of the month for all sessions.

In [ ]:
D = pd.DataFrame({"session": [s["session_id"] for s in S], "label": [s["label"] for s in S],
                  "start": [s["session_start_time"] for s in S]})
D["date"] = pd.to_datetime(D.start.str[:10])
print("Files on the first of a month:", (D.date.dt.day == 1).sum(), "of", len(D))
print(D[D.date.dt.day != 1].to_string())
print("Years:", dict(sorted(collections.Counter(D.date.dt.year).items())))
print("Time of day and offsets:", collections.Counter(D.start.str[10:]))
rows = []
for s in S:
    t0_ = s["trials"].start_time.min()
    if t0_ > 1e8:
        rows.append({"session": s["session_id"], "start": s["session_start_time"][:10],
                     "first_trial_utc": pd.to_datetime(t0_, unit="s")})
EP = pd.DataFrame(rows)
EP["same_year_month"] = EP.start.str[:7] == EP.first_trial_utc.dt.strftime("%Y-%m")
print(f"Sessions whose trial times are Unix timestamps: {len(EP)}; their first trial falls in the year and month of session_start_time in {EP.same_year_month.sum()}")
print(f"Days of the month these timestamps give (first trial, UTC): {sorted(EP.first_trial_utc.dt.day.unique().tolist())}")
print(EP[~EP.same_year_month].to_string())
print("Sessions of the same subject as session 68, and sessions 67 to 69:")
print(D[D.session.isin([67, 68, 69])].to_string())

**Result: partially reproduced.** 86 of 87 files start on the first of a month; session 48 starts on
2010-03-13. Session 68 (H44) is dated 2000-01-01, six years before the earliest other session (2006),
and between sessions dated 2013. The trial and event times of 38 files are Unix timestamps; they fall
in the year and month of session_start_time in all 38, and they give the day of each of these
recordings (first trials on 23 different days of the month), which the date field was set to hide.

## units-table

> The units table holds spike_times, waveform_mean_encoding, waveform_mean_recognition (sampled at
> 100 kHz), origClusterID, SNR, IsolDist and electrodes, with one electrode per neuron.

In [ ]:
need = ["spike_times", "waveform_mean_encoding", "waveform_mean_recognition", "origClusterID", "SNR", "IsolDist", "electrodes"]
present = collections.Counter()
for s in S:
    for c in need:
        present[c] += c in s["units_datasets"]
present["IsolationDist (in place of IsolDist)"] = sum("IsolationDist" in s["units_datasets"] for s in S)
print("Files holding each column:", dict(present), "of", len(S))
print("Units linked to exactly one electrode:", sum(len(e) == 1 for s in S for e in s["unit_electrodes"]), "of", n_units)
rates = np.concatenate([s["waveform_rate"] for s in S])
print("Waveform sampling rate (Hz):", dict(collections.Counter(rates)), "; expected 100000 +/- 10000")
print("Waveform shapes:", collections.Counter(s["waveform_mean_encoding_shape"][1] for s in S))
for c in ["SNR", "IsolationDist", "origClusterID"]:
    v = np.concatenate([s[c] for s in S]).astype(float)
    fin = np.isfinite(v)
    print(f"{c}: finite for {fin.sum()} of {len(v)} units (nan {np.isnan(v).sum()}, inf {np.isinf(v).sum()}); finite range {v[fin].min():.3g} to {v[fin].max():.3g}")
iso = np.concatenate([s["IsolationDist"] for s in S])
print("IsolationDist above 1e6:", int(np.sum(iso[np.isfinite(iso)] > 1e6)))
print("Sessions with any non-finite SNR or IsolationDist:", sum(not (np.isfinite(s["SNR"]).all() and np.isfinite(s["IsolationDist"]).all()) for s in S))

**Result: partially reproduced.** Every file holds the listed columns, with IsolDist named IsolationDist;
every unit links to one electrode; the waveform sampling rate is 98400 Hz in every file, within
tolerance of 100 kHz. The quality metrics are not finite for every unit: SNR is missing or infinite
for 86 of 1864 units (75 nan, 11 inf), IsolationDist is nan for 493, and two IsolationDist values
exceed 1e6 (largest 8.08e32). 84 of 87 files have a unit with a non-finite metric.

## electrodes-table

> The electrodes table gives each electrode's brain area, MNI coordinates, impedance, the filter applied
> before spike sorting (300 to 3000 Hz) and the original hardware channel.

In [ ]:
EL = pd.concat([s["electrodes"].assign(session=s["session_id"]) for s in S])
def empty(v):
    return v is None or (isinstance(v, float) and np.isnan(v)) or (isinstance(v, str) and v.strip() == "")
for c in ["location", "x", "y", "z", "imp", "filtering", "origChannel"]:
    e = EL[c].map(empty)
    print(f"{c:12s} empty for {e.sum():4d} of {len(EL)} electrodes, in {EL[e].session.nunique()} sessions")
print("Filtering values:", dict(collections.Counter(EL.filtering)))
print("Sessions with empty coordinates:", sorted(int(x) for x in EL[EL.x.map(empty)].session.unique()))
print("imp as stored, HDF5 dtype per file:", dict(collections.Counter(s["imp_dtype"] for s in S)))
print("imp values as read (repr, counts):", dict(collections.Counter(repr(v) for v in EL.imp)))
print("imp of the first 5 electrodes of session", S[0]["session_id"], ":", S[0]["electrodes"].imp.values[:5].tolist())

**Result: not reproduced.** Location, filtering (300-3000Hz) and origChannel are filled for all 887
electrodes, but impedance is empty for every electrode in every file, which the criterion counts as not
reproduced: the column is stored as float64 in all 87 files and every one of the 887 values is nan. The x, y, z coordinates are empty for 99 electrodes in 8 sessions.

## isi-criterion

> Clusters were accepted as units only when fewer than 3% of their interspike intervals were shorter than 3 ms.

In [ ]:
rows = []
for s in S:
    for i, sp in enumerate(s["spikes"]):
        rows.append({"session": s["session_id"], "unit": i, "n_spikes": len(sp),
                     "isi_sorted": rm.isi_violation_fraction(sp, sort=True),
                     "isi_stored": rm.isi_violation_fraction(sp, sort=False),
                     "unsorted": bool(np.any(np.diff(sp) < 0)), "dup_spikes": int(np.sum(np.diff(np.sort(sp)) == 0))})
I = pd.DataFrame(rows)
for c in ["isi_sorted", "isi_stored"]:
    print(f"{c}: below 3% for {np.sum(I[c] < 0.03)} of {len(I)} units ({100*np.mean(I[c] < 0.03):.2f}%); median {100*I[c].median():.2f}%, max {100*I[c].max():.1f}%")
print("Units with spike times not sorted in time:", I.unsorted.sum(), " in sessions", sorted(I[I.unsorted].session.unique()))
print("Units with repeated identical spike times:", (I.dup_spikes > 0).sum())
print("Units at or above 3% (sorted):")
print(I[I.isi_sorted >= 0.03].to_string())

**Result: partially reproduced.** With spike times sorted, 1858 of 1864 units (99.68%) have fewer than 3%
of intervals below 3 ms; four units are at 3.0 to 9.0% (sessions 66, 124, 126) and two units of session
58 at 50%, because every spike of these two is stored twice (1634 and 412 repeated times) and the train
is not sorted. Stored order gives 1860 of 1864. 13 units in 7 sessions have spike times not sorted in
time.

## fig5a-mtl-locations

> All neurons were recorded from the medial temporal lobe, in the amygdala and the hippocampus.

In [ ]:
loc = collections.Counter(s["electrodes"].location.values[e[0]] for s in S for e in s["unit_electrodes"])
print("Units by electrode location:", dict(loc))
ok = sum(v for k, v in loc.items() if "amygdala" in k.lower() or "hippocampus" in k.lower())
print(f"Units in the amygdala or hippocampus: {ok} of {sum(loc.values())}")

**Result: reproduced.** All 1864 units are on electrodes in the left or right amygdala (1092) or
hippocampus (772).

## Behaviour: ROC per session

AUC, z-ROC slope and accuracy by confidence, per session, as the helper describes (paper's code).

In [ ]:
rows = []
for s in S:
    r = rm.recog_trials(s)
    lab, conf = r.label.values, r.conf.values
    h, f = rm.cumulative_roc(lab, conf)
    hi, lo = rm.accuracy_fixed_split(lab, conf)
    dhi, dlo, mode = rm.accuracy_dynamic_split(lab, conf)
    mhi, mlo, _ = rm.accuracy_dynamic_split(lab, conf, matlab=True)
    a = rm.auc(h, f)
    rows.append({"session": s["session_id"], "label": s["label"], "site": rm.site(s["label"]), "auc": a,
                 "auc_reversed_labels": rm.auc(*rm.cumulative_roc(1 - lab, conf)),
                 "slope": rm.zroc_slope(h, f), "acc_high": hi, "acc_low": lo, "dyn_high": dhi, "dyn_low": dlo, "dyn_mode": mode,
                 "mat_high": mhi, "mat_low": mlo,
                 "H": h, "F": f})
B = pd.DataFrame(rows)
print(B.drop(columns=["H", "F"]).describe().round(3).to_string())

## fig5c-auc-all

> Subjects distinguished new from old images; the behavioural AUC averaged over all sessions is 0.74 ± 0.10.

In [ ]:
m, sd = B.auc.mean(), B.auc.std(ddof=1)
print(f"AUC over {len(B)} sessions: mean {m:.3f} (expected 0.74 +/- 0.10), SD {sd:.3f} (expected 0.10 +/- 0.01); range {B.auc.min():.3f} to {B.auc.max():.3f}")
print(f"Sessions with AUC above 0.5: {(B.auc > 0.5).sum()}; one-sample t-test against 0.5: p = {stats.ttest_1samp(B.auc, 0.5).pvalue:.2e}")
print(f"With the stated label coding (0 = old) the mean AUC would be {B.auc_reversed_labels.mean():.3f}")
dd = B[~B.session.isin([b for (a, b) in pairs])]
print(f"Without the second file of each pair sharing spike trains ({len(dd)} sessions): mean {dd.auc.mean():.3f}, SD {dd.auc.std(ddof=1):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for h, f in zip(B.H, B.F):
    ax[0].plot(np.r_[0, f, 1], np.r_[0, h, 1], color="0.8", lw=0.7)
ax[0].plot(np.r_[0, np.mean(np.stack(B.F), 0), 1], np.r_[0, np.mean(np.stack(B.H), 0), 1], "o-", color="C0")
ax[0].plot([0, 1], [0, 1], "k:")
ax[0].set(xlabel="false alarm rate", ylabel="hit rate", title="ROC per session (all points in [0, 1])")
ax[1].hist([B.auc[B.site == "CS"], B.auc[B.site == "TWH"]], bins=15, label=["CS", "TWH"], stacked=True)
ax[1].set(xlabel="AUC", ylabel="sessions", title=f"AUC, range {B.auc.min():.2f} to {B.auc.max():.2f}")
ax[1].legend()
plt.tight_layout()

**Result: reproduced.** The mean AUC over the 87 sessions is 0.753 with SD 0.094 (expected 0.74 and
0.10), and 86 sessions are above 0.5. With the paper's stated label coding the mean would be 0.247,
which is how the reversed labels show in behaviour.

## fig5c-auc-sites

> Performance was similar at the two study sites, with an average AUC of 0.74 ± 0.09 for CS sessions and
> 0.69 ± 0.13 for TWH sessions.

In [ ]:
g = B.groupby("site").auc.agg(["count", "mean", "std"])
print(g.round(3).to_string())
print("Expected: CS 0.74 +/- 0.09 (SD 0.09 +/- 0.009); TWH 0.69 +/- 0.13 (SD 0.13 +/- 0.013)")
w = stats.ttest_ind(B.auc[B.site == "CS"], B.auc[B.site == "TWH"], equal_var=False)
print(f"Welch's t-test CS against TWH: t = {w.statistic:.2f}, p = {w.pvalue:.3f}; difference of means {g.loc['CS','mean']-g.loc['TWH','mean']:.3f}")

**Result: reproduced.** CS sessions (78) have a mean AUC of 0.760 with SD 0.085, TWH sessions (9) 0.688
with SD 0.138; all four values are within tolerance. The sites do not differ by Welch's t-test
(p = 0.161).

## fig5b-roc-asymmetric

> The ROC curve is asymmetric (p < 0.05), as expected for declarative memory.

Criterion: z-ROC slope per session (calcZROCSlope), mean below 1 with a two-sided one-sample t-test
against 1 at p < 0.05; Wilcoxon and the pooled ROC's slope are variants.

In [ ]:
sl = B.slope.dropna()
print(f"z-ROC slope over {len(sl)} sessions ({B.slope.isna().sum()} without two distinct false alarm points): mean {sl.mean():.3f}, SD {sl.std():.3f}, median {sl.median():.3f}, range {sl.min():.2f} to {sl.max():.2f}")
t = stats.ttest_1samp(sl, 1.0)
print(f"t-test against 1: t = {t.statistic:.2f}, p = {t.pvalue:.2e}; sessions with slope below 1: {(sl < 1).sum()}")
wx = stats.wilcoxon(sl - 1)
print(f"Wilcoxon signed-rank against 1: p = {wx.pvalue:.2e}")
lab_all = np.concatenate([rm.recog_trials(s).label.values for s in S]); conf_all = np.concatenate([rm.recog_trials(s).conf.values for s in S])
old, new = rm.typecounters(lab_all, conf_all)
hp, fp = np.cumsum(old) / old.sum(), np.cumsum(new) / new.sum()
print(f"Pooled ROC (all trials): z-ROC slope {rm.zroc_slope(hp, fp):.3f}")

**Result: reproduced.** The mean z-ROC slope over 82 sessions is 0.782, below 1 by a two-sided t-test
(p = 4.3e-09); 65 sessions have a slope below 1, the Wilcoxon test gives p = 2.0e-08 and the pooled ROC
has slope 0.816. Five sessions have fewer than two distinct false alarm points and no slope.

## fig5d-confidence-accuracy

> Accuracy is significantly higher in high than in low confidence trials (P < 0.001, paired t test) across all sessions.

Readings: (a) fixed split (high 1 or 6, low 2 to 5; NO_behavior_evaluateSessionStatus.m), (b) the
dynamic split of the Python code (helper.dynamic_split). Sessions without responses in a group are left
out. The unpaired tests the code runs (ttest2 in MATLAB, Welch's in Python) are variants, and so is
the code's session inclusion (check_inclusion: AUC at least 0.6 and at most one unused level per side).

In [ ]:
for name, hcol, lcol in [("(a) fixed split", "acc_high", "acc_low"), ("(b) dynamic split", "dyn_high", "dyn_low"),
                           ("variant: dynamic split as NO_dynamic_lowHighSplit.m (mode 1 low old = 4 or 5)", "mat_high", "mat_low")]:
    x = B.dropna(subset=[hcol, lcol])
    pt = stats.ttest_rel(x[hcol], x[lcol])
    ut = stats.ttest_ind(x[hcol], x[lcol])
    wt = stats.ttest_ind(x[hcol], x[lcol], equal_var=False)
    print(f"{name}: {len(x)} sessions ({len(B)-len(x)} left out); high {x[hcol].mean():.1f}%, low {x[lcol].mean():.1f}%, "
          f"difference {np.mean(x[hcol]-x[lcol]):.1f} points; high > low in {(x[hcol] > x[lcol]).sum()} sessions")
    print(f"    paired t = {pt.statistic:.2f}, p = {pt.pvalue:.1e}; unpaired p = {ut.pvalue:.1e}; Welch p = {wt.pvalue:.1e}")
    for site_, gg in x.groupby("site"):
        print(f"    {site_}: {len(gg)} sessions, high {gg[hcol].mean():.1f}% (SD {gg[hcol].std():.1f}), low {gg[lcol].mean():.1f}% (SD {gg[lcol].std():.1f})")
print("Sessions left out under (a):", B[B[["acc_high", "acc_low"]].isna().any(axis=1)].session.tolist())
print("Dynamic split modes:", dict(collections.Counter(B.dyn_mode)))
incl = []
for s in S:
    conf = rm.recog_trials(s).conf.values
    cnt = np.array([np.sum(conf == k) for k in range(1, 7)])
    incl.append(not ((np.sum(cnt[:3] == 0) > 1) or (np.sum(cnt[3:] == 0) > 1)))
inc = B[np.array(incl) & (B.auc >= 0.6)].dropna(subset=["dyn_high", "dyn_low"])
print(f"Code's inclusion, dynamic split: {len(inc)} sessions, high {inc.dyn_high.mean():.1f}%, low {inc.dyn_low.mean():.1f}%, "
      f"paired p = {stats.ttest_rel(inc.dyn_high, inc.dyn_low).pvalue:.1e}, Welch p = {stats.ttest_ind(inc.dyn_high, inc.dyn_low, equal_var=False).pvalue:.1e}")

**Result: reproduced.** With the fixed split, accuracy is 80.1% for high and 60.9% for low confidence
over 84 sessions (paired t-test p = 3.2e-16); with the Python code's dynamic split 78.2% and 60.8% over
82 sessions (p = 1.3e-11). The dynamic split as the MATLAB code has it, which keeps response 5 as low
confidence, gives 78.3% and 60.4% over 84 sessions (p = 1.2e-13). The unpaired tests and the code's
session inclusion also give p below 1e-10.

## Single neurons: VS and MS tests

For every unit: firing rates in the recognition trials in the Methods' window (0.2 to 1.2 s) and the
code's (0.2 to 1.7 s); the VS one-way ANOVA across the stimCategory values; the MS bootstrap between
label-0 and label-1 trials, (i) all 50 and 50 trials (Methods) and (ii) correct trials only, as the
code's ms_test. The bootstrap uses its own generator (seed 20200304, a standard choice); the spread
over five other seeds is shown below. Nulls: the category labels, or the new/old labels, shuffled within
each session, category, label and response together (one permutation per session, applied to all its units; 10 permutations). The selection's false-positive rate is 5%, 93 units of 1864.

In [ ]:
def unit_tests(rng, shuffle_rng=None):
    rows = []
    for s in S:
        r = rm.recog_trials(s)
        on, cat, lab, conf = r.start_time.values, r.stimCategory.values, r.label.values, r.conf.values
        if shuffle_rng is not None:
            # permute the trials' category, label and response together, so that rates lose their trial labels
            perm = shuffle_rng.permutation(len(lab))
            cat, lab, conf = cat[perm], lab[perm], conf[perm]
        correct = ((conf <= 3) & (lab == 0)) | ((conf >= 4) & (lab == 1))
        for i, sp in enumerate(s["spikes"]):
            d = {"session": s["session_id"], "unit": i, "site": rm.site(s["label"]), "dup": (s["session_id"], i) in drop}
            for wn, w in [("M", rm.METHODS_WINDOW), ("C", rm.CODE_WINDOW)]:
                fr = rm.window_rates(sp, on, w)
                d["vs_" + wn] = rm.vs_p(fr, cat)
                d["ms_all_" + wn] = rm.bootstrap_p(fr[lab == 0], fr[lab == 1], rng)
                d["ms_cor_" + wn] = rm.bootstrap_p(fr[(lab == 0) & correct], fr[(lab == 1) & correct], rng)
            d["rate_M"] = rm.window_rates(sp, on, rm.METHODS_WINDOW).mean()
            rows.append(d)
    return pd.DataFrame(rows)

t1 = time.time()
U = unit_tests(np.random.default_rng(20200304))
print(f"Tests on {len(U)} units in {time.time()-t1:.0f} s")
print("Units with no spike in any recognition window (Methods window):", int((U.rate_M == 0).sum()))
print("Units whose ANOVA could not be computed (all rates equal):", int(U.vs_M.isna().sum()), "(Methods window),", int(U.vs_C.isna().sum()), "(code window)")
seeds = {}
for sd in [1, 2, 3, 4, 5]:
    u2 = unit_tests(np.random.default_rng(sd))
    seeds[sd] = {c: int((u2[c] < 0.05).sum()) for c in ["ms_all_M", "ms_cor_M", "ms_all_C", "ms_cor_C"]}
    seeds[sd]["dual_main"] = int(((u2.vs_M < 0.05) & (u2.ms_all_M < 0.05)).sum())
    seeds[sd]["dual_code"] = int(((u2.vs_C < 0.05) & (u2.ms_cor_C < 0.05)).sum())
print("MS and dual counts under five other bootstrap seeds:")
print(pd.DataFrame(seeds).T.to_string())
nulls = []
prng = np.random.default_rng(7)
for k in range(10):
    un = unit_tests(np.random.default_rng(100 + k), shuffle_rng=prng)
    nulls.append({c: int((un[c] < 0.05).sum()) for c in ["vs_M", "vs_C", "ms_all_M", "ms_cor_C"]})
NU = pd.DataFrame(nulls)
print(f"Shuffled-trial nulls (10 permutations): units at p < 0.05 (false-positive rate 5% gives {0.05*len(U):.0f})")
print(pd.DataFrame({"mean": NU.mean(), "min": NU.min(), "max": NU.max()}).to_string())

## vs-cells-proportion

> 343 of the 1863 cells (18.4%) are visually selective, by a one-way ANOVA (P < 0.05) of the firing rate in
> recognition trials, in a 1 s window starting 200 ms after stimulus onset, across the 5 visual categories.

In [ ]:
def show(col, label):
    a = U[col] < 0.05
    nd = U[~U.dup]
    print(f"{label}: {a.sum()} of {len(U)} ({100*a.mean():.1f}%); trains counted once: {(nd[col] < 0.05).sum()} of {len(nd)} ({100*(nd[col] < 0.05).mean():.1f}%)")
show("vs_M", "VS, Methods window (main)")
show("vs_C", "VS, code window")
print(f"Expected 343 +/- 34.3 cells and 18.4 +/- 1.84 %; null (trials shuffled) mean {NU.vs_M.mean():.1f} units (Methods window), {NU.vs_C.mean():.1f} (code window)")
print("VS by site (Methods window):", U.groupby("site").vs_M.apply(lambda p: f"{(p<0.05).sum()} of {len(p)}").to_dict())

**Result: reproduced.** With the Methods' window 332 of 1864 units (17.8%) are visually selective, within
tolerance of 343 (18.4%); the code's window gives 343 (18.4%). Shuffling the trials gives 86.8 units on
average, so the count is well above chance.

## ms-cells-proportion

> 146 of the 1863 cells (7.8%) are memory selective, by a two-tailed bootstrap comparison of mean firing
> rates (1,000 runs, P < 0.05) between the 50 novel and 50 familiar recognition trials in a 1 s window
> starting 200 ms after stimulus onset.

In [ ]:
show("ms_all_M", "MS, Methods window, all trials (main)")
show("ms_cor_M", "MS, Methods window, correct trials")
show("ms_all_C", "MS, code window, all trials")
show("ms_cor_C", "MS, code window, correct trials (the code's choices)")
print("Expected 146 +/- 14.6 cells and 7.8 +/- 0.78 %")
print("Null (trials shuffled): mean %.1f units (main), %.1f (code's choices)" % (NU.ms_all_M.mean(), NU.ms_cor_C.mean()))

**Result: not reproduced.** With the Methods' window and all 50 and 50 trials, 112 units (6.0%) are memory
selective, below the tolerance of 146 (131.4 to 160.6); five other seeds give 108 to 117. The
shuffled-trial null gives 92.4 on average. With the paper's code's choices, correct trials only and the
0.2 to 1.7 s window, 151 units (8.1%) are selected, within tolerance, and 141 to 153 over the five
seeds. The paper's numbers follow its code, not its Methods text.

## dual-cells-proportion

> 26 cells (1.4%) are both visually and memory selective.

In [ ]:
for name, v, m_ in [("main (Methods window, all trials)", "vs_M", "ms_all_M"), ("code's choices (1.7 s window, correct trials)", "vs_C", "ms_cor_C"),
                    ("Methods window, correct trials", "vs_M", "ms_cor_M")]:
    d = (U[v] < 0.05) & (U[m_] < 0.05)
    print(f"Dual, {name}: {d.sum()} of {len(U)} ({100*d.mean():.2f}%); trains counted once {(d & ~U.dup).sum()}")
print("Expected 26 +/- 2.6 cells and 1.4 +/- 0.14 %")

**Result: not reproduced.** With the main tests 12 units (0.64%) are both VS and MS, against 26 (1.4%).
With the code's choices exactly 26 units (1.39%) are, and 24 to 26 over five other seeds.

## fig5f-site-proportions

> The percentages of VS, MS and dual cells are comparable between CS sessions (18.4%, 7.7% and 1.4%) and
> the new TWH sessions (19.4%, 10.4% and 1.3%).

Criterion: two-sided Fisher's exact test between sites for each cell type; at least four of the six
values within tolerance.

In [ ]:
expected = {("CS", "VS"): 18.4, ("CS", "MS"): 7.7, ("CS", "dual"): 1.4, ("TWH", "VS"): 19.4, ("TWH", "MS"): 10.4, ("TWH", "dual"): 1.3}
for name, v, m_ in [("main (Methods window, all trials)", "vs_M", "ms_all_M"), ("code's choices", "vs_C", "ms_cor_C")]:
    sel = {"VS": U[v] < 0.05, "MS": U[m_] < 0.05, "dual": (U[v] < 0.05) & (U[m_] < 0.05)}
    print(name)
    within = 0
    for typ, a in sel.items():
        cnt = {st: (int(a[U.site == st].sum()), int((U.site == st).sum())) for st in ["CS", "TWH"]}
        tab_ = [[cnt["CS"][0], cnt["CS"][1] - cnt["CS"][0]], [cnt["TWH"][0], cnt["TWH"][1] - cnt["TWH"][0]]]
        p = stats.fisher_exact(tab_).pvalue
        line = []
        for st in ["CS", "TWH"]:
            pc = 100 * cnt[st][0] / cnt[st][1]
            ok = abs(pc - expected[(st, typ)]) <= 0.1 * expected[(st, typ)] + 1e-9
            within += ok
            line.append(f"{st} {cnt[st][0]}/{cnt[st][1]} = {pc:.1f}% (exp {expected[(st, typ)]}, {'within' if ok else 'outside'})")
        print(f"  {typ}: " + "; ".join(line) + f"; Fisher p = {p:.3f}")
    print(f"  values within tolerance: {within} of 6")

**Result: partially reproduced.** With the main tests no Fisher test finds a difference between the sites
(p = 0.360, 0.133, 0.398), but only 3 of the 6 percentages are within tolerance: CS 17.6, 5.8 and 0.6%,
TWH 22.1, 10.4 and 1.3%. With the code's choices all six are within tolerance (CS 18.4, 8.0, 1.4%;
TWH 19.5, 10.4, 1.3%) and no test finds a difference.

## twh-sessions

> The dataset includes 9 sessions from a new institution, Toronto Western Hospital.

In [ ]:
twh = F[F.label.str.startswith("T")]
print(f"Sessions with a subject label starting with T: {len(twh)} (expected 9 +/- 0.9), subjects {twh.label.nunique()}, units {twh.units.sum()}")
print(twh[["session", "label", "units"]].to_string())
print("Institution field of these files:", collections.Counter(SID[k]["institution"] for k in twh.session))

**Result: reproduced.** Nine sessions, of 8 subjects with 77 units, have labels starting with T, and their
institution field is Toronto Western Hospital.

## previous-release-link

> The sessions of the earlier release (Faraut et al. 2018) are found in this dandiset under the same
> subject, session id and task variant.

Matched by session id; a match agrees when the subject label and the variant (from the image
directory, Table 2 of Faraut et al.) are the table's.

In [ ]:
fv = F.set_index("session")
rows = []
for r in faraut_t1.itertuples():
    found = r.session_id in fv.index
    d = {"session": r.session_id, "label": r.label, "variant": r.variant, "age": r.age, "sex": r.sex, "found": found}
    if found:
        x = fv.loc[r.session_id]
        sj = SID[r.session_id]["subject"]
        d.update({"file_label": x.table_label, "file_variant": x.variant, "file_age": sj.get("age"), "file_sex": sj.get("sex")})
    rows.append(d)
LK = pd.DataFrame(rows)
LK["agrees"] = LK.found & (LK.label == LK.file_label) & (LK.variant == LK.file_variant)
print(f"Sessions in Faraut et al. Table 1: {len(LK)}; found by session id: {LK.found.sum()}; agree on subject and variant: {LK.agrees.sum()} ({100*LK.agrees.mean():.1f}%)")
print("Not found or not agreeing:")
print(LK[~LK.agrees].to_string())
m_ = LK[LK.found]
print(f"Age equal: {(m_.age.astype(str) == m_.file_age.astype(str)).sum()} of {len(m_)}; sex equal: {(m_.sex == m_.file_sex).sum()}")
print(m_[(m_.age.astype(str) != m_.file_age.astype(str)) | (m_.sex != m_.file_sex)][["session", "label", "age", "file_age", "sex", "file_sex"]].to_string())

**Result: partially reproduced.** Of the 65 sessions in Table 1 of Faraut et al. 2018, 64 (98.5%) are in
the dandiset under the same subject and variant. Session 98 (H48, variant 2) is not in it, nor in
Online-only Table 1. Of the matched sessions, age differs for 4 (sessions 52, 59, 60, 99) and sex for 1
(99), the same three subjects that differ from Online-only Table 1.

## previous-release-additions

> Relative to the earlier release of this task's data (Faraut et al. 2018), this release adds 23
> sessions, 17 subjects and 288 neurons.

In [ ]:
old_sessions = set(faraut_t1.session_id)
old_labels = set(faraut_t1.label)
new = F[~F.session.isin(old_sessions)]
new_subj = set(F.table_label) - old_labels
print(f"New sessions: {len(new)} (expected 23 +/- 2.3): {new.session.tolist()}")
print(f"New subjects: {len(new_subj)} (expected 17 +/- 1.7): {sorted(new_subj)}")
print(f"New neurons (unit rows of new sessions): {new.units.sum()} (expected 288 +/- 28.8); trains counted once: {new.units.sum() - sum(1 for d in drop if d[0] in set(new.session))}")
t3 = rm.pmc_table(rm.FARAUT_PMC, "Table 3")
faraut_cells = {r[0]: int(r[1]) for r in t3 if r and r[0] in ("Hippocampus", "Amygdala")}
nf = sum(faraut_cells.values())
print("Faraut et al. Table 3, cells recorded:", faraut_cells, "total", nf)
print(f"Variant, differences of totals: sessions {len(F)} - {len(faraut_t1)} = {len(F)-len(faraut_t1)}; "
      f"subjects {F.table_label.nunique()} - {faraut_t1.label.nunique()} = {F.table_label.nunique()-faraut_t1.label.nunique()}; "
      f"neurons {n_units} - {nf} = {n_units-nf}")
old_units = F[~F.session.isin(new.session)].units.sum()
print(f"Unit rows of the {len(F)-len(new)} sessions of the earlier release in this dandiset: {old_units}, against {nf} in Faraut et al.")
print("Sessions of new subjects:", F[F.table_label.isin(new_subj)].session.tolist())

**Result: partially reproduced.** The dandiset has 23 sessions and 17 subjects that are not in the
earlier release, as stated, but the new sessions hold 325 unit rows, outside the tolerance of 288. The
difference of the totals is exactly 288 (1864 - 1576), so the 64 earlier sessions hold 1539 units here
against the 1576 cells of Faraut et al. Table 3: the paper's 288 matches a difference of totals, not
the units of the new sessions.

## fig4-example-vs-neuron

> Neuron 132-177-4-RA (session 132, channel 177, cell 4, right amygdala) is visually selective, responding most to images of animals.

In [ ]:
s = SID[132]
el = s["electrodes"]
print(el[["origChannel", "location"]].to_string())
cand = [i for i, e in enumerate(s["unit_electrodes"]) if el.origChannel.values[e[0]] == 177]
print("Units on channel 177:", cand, " electrode rows:", [int(s["unit_electrodes"][i][0]) for i in cand],
      " origClusterID:", [int(s["origClusterID"][i]) for i in cand])
r = rm.recog_trials(s)
for i in cand:
    print(f"Unit {i}: location {el.location.values[s['unit_electrodes'][i][0]]}, {len(s['spikes'][i])} spikes")
    for wn, w in [("Methods 0.2-1.2 s", rm.METHODS_WINDOW), ("code 0.2-1.7 s", rm.CODE_WINDOW)]:
        fr = rm.window_rates(s["spikes"][i], r.start_time.values, w)
        means = pd.Series(fr).groupby(r.category_name.values).mean().sort_values(ascending=False)
        print(f"  {wn}: ANOVA p = {rm.vs_p(fr, r.stimCategory.values):.2e}; mean rate by category (Hz): " +
              ", ".join(f"{k} {v:.2f}" for k, v in means.items()))
print("Category names and ids in this session:", sorted(set(zip(r.stimCategory, r.category_name))))

**Result: reproduced.** Session 132 has one unit on channel 177, unit 4 on electrode row 4, in the right
amygdala. Its rate differs across the 5 categories (ANOVA p = 1.2e-22, Methods' window; 7.9e-19, code's
window), and it fires most to animals (6.75 Hz against at most 0.85 Hz for the other categories).

## fig3-example-vs-neuron

> Neuron 144-140-3-RH (session 144, channel 140, cell 3, right hippocampus) is visually selective for phones.

Session 144 is not among the dandiset's session ids, nor in Online-only Table 1. The paper's own deposit on
OSF (doi:10.17605/OSF.IO/HV7JA) is searched below by listing the NWB files in its NWBData.zip. Figure 3
shows the Python export and import pipeline ("Illustration of analysis pipeline in Python. Shown is how
spike times and trial information of the native dataset is exported into NWB (left) and how this data
is subsequently read from NWB for plotting (right)"). The scripts it shows are in the paper's code,
python/export/demo/export.py and import.py at commit 1d95162; they name the session (NOID 144) as
TWH103_111918, a session of subject TWH103, and import.py plots `vs_neurons[2]`, the third unit of the
file that passes vs_test (0.2 to 1.7 s, p <= 0.05). This attempt takes session 144 to be the dandiset's
only TWH103 session, and cell 3 to be that third VS unit, as an assumption; the other unit on channel 140
is shown as the alternative reading.

In [ ]:
import requests, zipfile, remfile
print("Session 144 in the dandiset:", 144 in SID, "; in Online-only Table 1:", 144 in set(paper_t1.session_id),
      "; in Faraut et al. Table 1:", 144 in set(faraut_t1.session_id))
osf = rm.retry(requests.get, "https://api.osf.io/v2/nodes/hv7ja/files/osfstorage/", timeout=120).json()
for x in osf["data"]:
    name = x["attributes"]["name"]
    if name == "NWBData.zip":
        url = rm.retry(requests.head, x["links"]["download"], allow_redirects=True, timeout=120).url
        names = [n for n in rm.retry(lambda: zipfile.ZipFile(remfile.File(url)).namelist()) if n.endswith(".nwb")]
        ids = sorted(int(n.split("NOID")[1].split(".")[0]) for n in names)
        print(f"OSF NWBData.zip: {len(names)} NWB files; session 144 among them: {144 in ids}; same session ids as the dandiset: {set(ids) == set(SID)}")

In [ ]:
# Category names per session, and units on channel 140 in the right hippocampus, with their preferred category.
cats = pd.DataFrame([{"session": s["session_id"], "variant": F.set_index("session").variant[s["session_id"]],
                      "categories": tuple(sorted(set(s["trials"].category_name)))} for s in S])
print("Category name sets (number of sessions, variants, sessions):")
for k, g in cats.groupby("categories"):
    print(f"  {k}: {len(g)} sessions, variants {sorted(g.variant.unique().tolist())}; sessions {g.session.tolist()}")
print("Sessions with a category whose name contains 'phone':", cats[cats.categories.map(lambda c: any("phone" in x.lower() for x in c))].session.tolist())
rows = []
for s in S:
    el = s["electrodes"]
    r = rm.recog_trials(s)
    for i, e in enumerate(s["unit_electrodes"]):
        if el.origChannel.values[e[0]] == 140 or "right hippocampus" in str(el.location.values[e[0]]).lower():
            fr = rm.window_rates(s["spikes"][i], r.start_time.values, rm.METHODS_WINDOW)
            means = pd.Series(fr).groupby(r.category_name.values).mean()
            rows.append({"session": s["session_id"], "unit": i, "channel": int(el.origChannel.values[e[0]]),
                         "location": el.location.values[e[0]], "vs_p": rm.vs_p(fr, r.stimCategory.values),
                         "preferred": means.idxmax()})
RH = pd.DataFrame(rows)
print(f"Units on channel 140 (any location): {(RH.channel == 140).sum()}")
print(RH[RH.channel == 140].to_string())
rhvs = RH[RH.location.str.lower().str.contains("right hippocampus") & (RH.vs_p < 0.05)]
print(f"Right-hippocampus units: {RH.location.str.lower().str.contains('right hippocampus').sum()}; VS (Methods window): {len(rhvs)}; preferred category among these:",
      dict(collections.Counter(rhvs.preferred)))

In [ ]:
base = ("https://raw.githubusercontent.com/rutishauserlab/recogmem-release-NWB/1d95162d2672e71dd234a31e4b8d70017a289ee3/"
        "RutishauserLabtoNWB/events/newolddelay/python/export/demo/")
for name in ["export.py", "import.py"]:
    txt = rm.retry(requests.get, base + name, timeout=60).text
    print(f"{name} at commit 1d95162:", [l.strip() for l in txt.splitlines() if "NOID" in l or "TWH103" in l or "vs_neurons[" in l])
print("Sessions of subject T103 (TWH103) in the dandiset:", F[F.label == "T103"][["session", "identifier", "start"]].values.tolist())
s = SID[2009]
el, r = s["electrodes"], rm.recog_trials(s)
rows = []
for i, e in enumerate(s["unit_electrodes"]):
    fm = rm.window_rates(s["spikes"][i], r.start_time.values, rm.METHODS_WINDOW)
    fc = rm.window_rates(s["spikes"][i], r.start_time.values, rm.CODE_WINDOW)
    means = pd.Series(fm).groupby(r.category_name.values).mean().sort_values(ascending=False)
    rows.append({"unit": i, "unit_id": int(s["unit_ids"][i]), "channel": int(el.origChannel.values[e[0]]), "location": el.location.values[e[0]],
                 "origClusterID": int(s["origClusterID"][i]), "spikes": len(s["spikes"][i]),
                 "vs_p_code": rm.vs_p(fc, r.stimCategory.values), "vs_p_methods": rm.vs_p(fm, r.stimCategory.values),
                 "rates_methods_Hz": ", ".join(f"{k} {v:.2f}" for k, v in means.items())})
V = pd.DataFrame(rows)
print(V.to_string())
vs_order = V[V.vs_p_code <= 0.05].unit.tolist()
print("Units passing the code's vs_test (0.2 to 1.7 s, p <= 0.05), in file order:", vs_order, "; vs_neurons[2] is unit", vs_order[2] if len(vs_order) > 2 else None)
for u in V[V.channel == 140].unit:
    x = V.set_index("unit").loc[u]
    print(f"Unit {u} (channel 140, {x.location}): Methods-window ANOVA p = {x.vs_p_methods:.2e}, code window p = {x.vs_p_code:.2e}; highest rate: {x.rates_methods_Hz.split(',')[0]}")

**Result: reproduced, under the assumption that links session 144 to session 2009.** The paper's
Figure 3 scripts (export.py and import.py at commit 1d95162) name session 144 as TWH103_111918, and the
dandiset holds one TWH103 session, 2009, dated 2018-11-01. It has two units on channel 140, both in the
right hippocampus. Unit 2, the third unit that passes the code's vs_test and so the one import.py plots,
has a Methods-window ANOVA p = 1.3e-09 and fires most to phones (9.10 Hz against at most 2.70 Hz). Under
the other reading of cell 3, unit 1 is also selective (p = 0.027) but fires most to small animals
(1.40 Hz). Session 144 itself is not in the dandiset or in the paper's OSF deposit.

## Summary

| Claim | Outcome |
|---|---|
| counts-neurons-subjects | reproduced |
| counts-sessions | reproduced |
| table1-demographics | partially reproduced |
| task-structure | partially reproduced |
| new-old-label-coding | not reproduced |
| ttl-codes | partially reproduced |
| experiment-ids | partially reproduced |
| session-dates | partially reproduced |
| units-table | partially reproduced |
| electrodes-table | not reproduced |
| isi-criterion | partially reproduced |
| fig5a-mtl-locations | reproduced |
| fig5c-auc-all | reproduced |
| fig5c-auc-sites | reproduced |
| fig5b-roc-asymmetric | reproduced |
| fig5d-confidence-accuracy | reproduced |
| vs-cells-proportion | reproduced |
| ms-cells-proportion | not reproduced |
| dual-cells-proportion | not reproduced |
| fig5f-site-proportions | partially reproduced |
| twh-sessions | reproduced |
| previous-release-link | partially reproduced (with Faraut et al. 2018) |
| previous-release-additions | partially reproduced (with Faraut et al. 2018) |
| fig4-example-vs-neuron | reproduced |
| fig3-example-vs-neuron | reproduced (assumed: session 144 is session 2009) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: the behaviour and the cell selections follow the paper's
code (see `recogmem.py`); the MS and dual claims are judged on the Methods' description (1 s window,
all trials), as the claims file fixes, and the code's choices (1.5 s window, correct trials only), which
give the paper's numbers, are shown as variants. Firing rates use the trial's start_time as stimulus
onset, as the code does. The bootstrap seed is a standard choice; the spread over seeds is shown. The
Figure 3 neuron is taken from session 2009, the session the paper's Figure 3 scripts name as 144, an
assumption.

What was found about the dataset:

- new_old_labels_recog is 1 for old and 0 for new stimuli, the reverse of the paper's text and of the
  column's own description.
- The electrodes' impedance is empty in every file; coordinates are empty in 8 sessions.
- SNR or IsolationDist is not finite for many units, and IsolDist is named IsolationDist.
- Three subjects have an age or sex that differs from both papers' patient tables.
- 8 sessions have 50 or 97 encoding trials, not 100.
- Four files use experiment ids outside Table 2, and nine hold event codes outside Table 1.
- 38 files store times as Unix timestamps, which give the exact day of recording that
  session_start_time was set to hide; session 48 is dated on the 13th and session 68 in 2000.
- stop_time is the end of the trial, not the stimulus offset as the paper describes; stim_off_time
  holds the offset.
- The same spike trains are in two files for sessions 38 and 39 (11 units), 43 and 44 (23) and 54 and
  55 (32): sessions recorded in one continuous recording, whose units are the same neurons.
- Two units of session 58 store every spike twice, and 13 units have spike times out of order.
- Session 144, of the Figure 3 example neuron, is in neither the dandiset nor the paper's OSF deposit under
  that id; the paper's code names it as a session of TWH103, whose only session here is 2009.